In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans

# ========= 1) Excel 读列 =========
file_path = r"your_data.xlsx"
sheet_name = "Discretize"
series_col = "value"
df = pd.read_excel(file_path, sheet_name=sheet_name)
s = df[series_col]

# ========= 2) 参数模板 =========
params = {
    "n_bins": 4  # int: 分箱数
}

equal_width = pd.cut(s, bins=params["n_bins"], labels=False)
equal_freq = pd.qcut(s, q=params["n_bins"], labels=False, duplicates="drop")
kmeans_label = KMeans(n_clusters=params["n_bins"], random_state=42, n_init=10).fit_predict(s.to_numpy().reshape(-1,1))

print(equal_width.head(), equal_freq.head(), kmeans_label[:10])


In [ ]:
"""
等宽法、等频法、聚类离散化

使用方法：
1. 按照下方 TODO 修改 DATA_FILE、列名、参数和输出文件名。
2. 将数据文件放在本脚本同目录，或把 DATA_FILE 改成绝对路径。
3. 运行：python "等宽法、等频法、聚类离散化.py"
"""

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans



DATA_FILE = "data.csv"  # TODO: 请填写[数据文件路径]，说明：CSV/Excel 均可；若使用 Excel，请在 load_data 中改为 read_excel。
OUTPUT_FILE = "model_output.csv"  # TODO: 请填写[输出文件名]，说明：保存模型结果，建议保留 .csv 或 .xlsx 后缀。
RANDOM_STATE = 42  # TODO: 请填写[随机种子]，说明：用于复现实验；整数即可。
TARGET_COLUMN = "x"  # TODO: 请填写[待离散化列名]，说明：连续数值列。
BIN_COUNT = 4  # TODO: 请填写[分箱数/聚类数]，说明：正整数，过大会导致每箱样本过少。



REQUIRES_DATA = True  # 参数型模型可不提供数据文件；表格型模型必须提供数据。


def load_data() -> pd.DataFrame:
    """读取用户数据；竞赛时通常把 Excel/CSV 表格整理成一行一个样本。"""
    path = Path(DATA_FILE)
    if not path.exists():
        if not REQUIRES_DATA:
            return pd.DataFrame()
        raise FileNotFoundError(
            f"未找到数据文件 {DATA_FILE}。请先修改 DATA_FILE，或将数据放到脚本同目录。"
        )
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    return pd.read_csv(path)


def run_model(data: pd.DataFrame) -> None:
    result = data.copy()
    values = result[TARGET_COLUMN].to_numpy().reshape(-1, 1)
    result["等宽分箱"] = pd.cut(result[TARGET_COLUMN], bins=BIN_COUNT, labels=False)
    result["等频分箱"] = pd.qcut(result[TARGET_COLUMN], q=BIN_COUNT, labels=False, duplicates="drop")
    result["聚类离散化"] = KMeans(n_clusters=BIN_COUNT, random_state=RANDOM_STATE, n_init="auto").fit_predict(values)
    result.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(result.head())


if __name__ == "__main__":
    df = load_data()
    run_model(df)
